## Analysing SegFormer segmentation model
---

In this notebook, we are going to fine-tune SegFormerForSemanticSegmentation on a custom semantic segmentation dataset. In semantic segmentation, the goal for the model is to label each pixel of an image with one of a list of predefined classes.

## Imports 
---

In [ ]:
#external
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm 
import albumentations as A
import random
SEED = 42
random.seed(SEED)

#model
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import torch
import pytorch_lightning as pl

from torch.utils.data import Dataset, DataLoader

from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks.model_checkpoint import ModelCheckpoint

#metrics
import evaluate
from torch import nn

#utils
from src.utils.dataset import load_foodseg103_splits
from src.utils.visualization import predict_random_images

#constants
from src.constants.category_id import CATEGORY_ID

## Testing on custom dataset
---

In [ ]:
IMAGE_SIZE = 512
LEARNING_RATE = 5e-5
BATCH_SIZE = 2
NUM_EPOCHS = 20
MODEL_NAME = "nvidia/mit-b4"

### Defining Dataset

In [ ]:
class SemanticSegmentationFoodDataset(Dataset):
    def __init__(self, dataset:pd.DataFrame, transform:A.Compose=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return self.dataset.shape[0]
    
    def __getitem__(self, index):
        original_image = np.array(self.dataset[index]["image"])
        mask = np.array(self.dataset[index]["label"])
        transformed = self.transform(image=original_image, mask=mask)
        return transformed["image"], transformed["mask"]

def check_image_mask_shape(df):
    wrong_indices = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        image = decode_image_from_bytes(row["image"])
        mask = decode_image_from_bytes(row["label"])
        if image.shape[:2] != mask.shape[:2]:
            wrong_indices.append(idx)
    return wrong_indices

In [ ]:
df = load_foodseg103_splits()

In [ ]:
train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),  # Adiciona alguma variabilidade extra
    A.RandomBrightnessContrast(p=0.4),
    A.ColorJitter(p=0.3),  # Mudança de cor, saturação, etc
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=20, p=0.5),

    A.GaussianBlur(blur_limit=(3,5), p=0.1),  # Um pouco de desfoque
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    A.pytorch.ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    A.pytorch.ToTensorV2()
])

In [ ]:
train_dataset = SemanticSegmentationFoodDataset(df["train"], train_transform)
test_dataset = SemanticSegmentationFoodDataset(df["test"], val_transform)
val_dataset = SemanticSegmentationFoodDataset(df["validation"], val_transform)

In [ ]:
print(f"Number of training examples: {train_dataset.__len__()}")
print(f"Number of testing examples: {test_dataset.__len__()}")
print(f"Number of validation examples: {val_dataset.__len__()}")

In [ ]:
image_processor = SegformerImageProcessor.from_pretrained(MODEL_NAME)
image_processor.do_reduce_labels = False
image_processor.do_normalize = False
image_processor.do_resize = False
image_processor.do_rescale = False

In [ ]:
def collate_fn(batch):    
    inputs = list(zip(*batch))
    images = inputs[0]
    segmentation_maps = inputs[1]
    batch = image_processor(
        images,
        segmentation_maps=segmentation_maps,
        return_tensors="pt",
    )
    return batch

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=False)

### Model definition

In [ ]:
class SegformerFinetuner(pl.LightningModule):
    def __init__(self, train_dataloader, val_dataloader, learning_rate=0.00005):
        super().__init__()
        self.train_dl = train_dataloader
        self.val_dl = val_dataloader
        self.learning_rate = learning_rate
        
        self.num_classes = len(CATEGORY_ID)
        self.model = SegformerForSemanticSegmentation.from_pretrained(
            MODEL_NAME, 
            num_labels=self.num_classes,
            id2label=CATEGORY_ID,
            label2id={v:k for k,v in CATEGORY_ID.items()},
            ignore_mismatched_sizes=True #number of classes is diff from original model
        )
        for param in self.model.base_model.parameters():
            param.requires_grad = False
        for block in self.model.base_model.encoder.block[-3:]:
            for param in block.parameters():
                param.requires_grad = True

        self.val_mean_iou = evaluate.load("mean_iou")
        self.train_mean_iou = evaluate.load("mean_iou")
        self.validation_step_outputs = []
            
    def forward(self, images, masks):
        return self.model(pixel_values=images, labels=masks)
    
    def training_step(self, batch, batch_idx):
        images, masks = batch['pixel_values'], batch['labels']
        outputs = self(images, masks)
        loss, logits = outputs.loss, outputs.logits
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        upsampled_logits = nn.functional.interpolate(
            logits, 
            size=masks.shape[-2:], 
            mode="bilinear", 
            align_corners=False
        )
        predicted = upsampled_logits.argmax(dim=1)
        self.train_mean_iou.add_batch(
            predictions=predicted.detach().cpu().numpy(), 
            references=masks.detach().cpu().numpy()
        )
        return loss
    
    def validation_step(self, batch, batch_idx):
        images, masks = batch['pixel_values'], batch['labels']
        outputs = self(images, masks)
        loss, logits = outputs.loss, outputs.logits
        self.validation_step_outputs.append(loss)
        upsampled_logits = nn.functional.interpolate(
            logits, 
            size=masks.shape[-2:], 
            mode="bilinear", 
            align_corners=False
        )
        predicted = upsampled_logits.argmax(dim=1)
        self.val_mean_iou.add_batch(
            predictions=predicted.detach().cpu().numpy(), 
            references=masks.detach().cpu().numpy()
        )
        return loss
    
    def on_train_epoch_end(self): 
        metrics = self.train_mean_iou.compute(
              num_labels=self.num_classes, 
              ignore_index=255, 
              reduce_labels=False,
        )
        train_mean_iou = metrics["mean_iou"]
        train_mean_accuracy = metrics["mean_accuracy"]
        self.log("train_mean_iou", train_mean_iou, on_epoch=True, prog_bar=True, logger=True)
        self.log("train_mean_accuracy", train_mean_accuracy, on_epoch=True, prog_bar=True, logger=True)
        return metrics
    
    def on_validation_epoch_end(self): 
        metrics = self.val_mean_iou.compute(
              num_labels=self.num_classes, 
              ignore_index=255, 
              reduce_labels=False,
        )
        avg_val_loss = torch.stack(self.validation_step_outputs).mean()
        val_mean_iou = metrics["mean_iou"]
        val_mean_accuracy = metrics["mean_accuracy"]
        metrics = {"avg_val_loss": avg_val_loss, "val_mean_iou":val_mean_iou, "val_mean_accuracy":val_mean_accuracy}
        self.log_dict(metrics, on_epoch=True, prog_bar=True, logger=True)
        self.validation_step_outputs.clear() #free memory
        return metrics

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            [p for p in self.parameters() if p.requires_grad],
            lr=self.learning_rate,
            weight_decay=0.02
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',              
            factor=0.1,              
            patience=2,              
            verbose=True             
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'avg_val_loss',  
                'interval': 'epoch',
                'frequency': 1,
            }
        }
    
    def train_dataloader(self):
        return self.train_dl
    
    def val_dataloader(self):
        return self.val_dl

segformer_finetuner = SegformerFinetuner.load_from_checkpoint(
    "src/model/nvidia-mit-b3-512x512-epoch=8-val_mean_iou=0.4038-train_mean_iou=0.7578.ckpt",
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader)

segformer_finetuner.model.save_pretrained("src/model/segformer-b3-finetuned-foodseg103")

In [ ]:
segformer_finetuner = SegformerFinetuner(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    learning_rate=LEARNING_RATE
)

In [ ]:
early_stop_callback = EarlyStopping(
    monitor="val_mean_iou",
    min_delta=0,
    patience=3, 
    mode="max"
)

model_name_formatted = MODEL_NAME.replace('/', '-')
image_size_formatted = f"{IMAGE_SIZE}x{IMAGE_SIZE}"

checkpoint_callback = ModelCheckpoint(
    dirpath="./src/model",
    filename=f"{model_name_formatted}-{image_size_formatted}-{{epoch}}-{{val_mean_iou:.4f}}-{{train_mean_iou:.4f}}",
    save_top_k=1,
    monitor="val_mean_iou",
    mode="max"
)

pl.seed_everything(SEED, workers=True)
trainer = pl.Trainer(
    callbacks=[early_stop_callback, checkpoint_callback],
    max_epochs=NUM_EPOCHS
)

In [ ]:
trainer.fit(segformer_finetuner)

%reload_ext tensorboard
%tensorboard --logdir lightning_logs/version_12/ --port 6007

### Predict

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = segformer_finetuner.model
model.to(device)
model.eval();

In [ ]:
predict_random_images(model, val_transform, test_dataset, num_images=20)

%load_ext tensorboard
%tensorboard --logdir lightning_logs/

## References
[1] https://github.com/NielsRogge/Transformers-Tutorials/tree/master/SegFormer

[2] https://blog.roboflow.com/how-to-train-segformer-on-a-custom-dataset-with-pytorch-lightning/#create-a-dataset

[3] https://github.com/3lc-ai/3lc-examples/blob/main/example-notebooks/pytorch-lightning-segformer.ipynb